In [2]:
#离线阶段
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain.chat_models import init_chat_model
from app.core.config import settings
# 在线阶段
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

# =============1.准备文档================
with open('../resources/评估.md', 'r', encoding='utf-8') as f:
    doc = f.read()

# =============2.切分文档================
# 切分依据，这里是按照三级标题
headers_to_split_on = [
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3"),
]

# 创建切分器
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on,
    strip_headers=False # 在正文中是否去除标题，False，不去除
)

# 切分文档, 返回的是 list[Document]
docs = markdown_splitter.split_text(doc)

for i, doc in enumerate(docs, start=1):
    doc.id = f'doc_{i}'

# 1.初始化向量模型
ollama_embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024  # 可选：减少维度以节省存储
)

# 2.初始化向量数据库
vector_store = InMemoryVectorStore(embedding=ollama_embeddings)
# 保存文档到向量数据库
ids = vector_store.add_documents(docs)

# 1.初始化模型
model = init_chat_model(
    "deepseek-v4-flash",
    api_key=settings.llm.api_key,
    extra_body={'thinking': {'type': 'disabled'}}
)

# 2.定义RAG函数
def rag_chat(user_input: str):
    # 1.拿到用户问题
    print("================User Message================")
    print(user_input, "\n")

    # 2.检索知识片段, lisi[Document]
    docs = vector_store.similarity_search(user_input, k=2)
    context = "\n\n".join(doc.page_content for doc in docs)
    print('=============== Context ====================')
    print(context, "\n")

    # 3.拼接新提示词
    prompt = f'''
    你是一个教资考试专家，你根据文档回答用户问题，文档中没有的不要编造，直接说不知道。
    回答直接引用文档中原文，不要自己发挥！
    context: {context}
    query: {user_input}
    '''

    # 4.调用模型
    result = model.invoke(prompt)
    print("=================AI Message================")
    print(result.content)

# 测试
rag_chat("论语中教育的目的是什么？")
rag_chat("你好？")
##可以发现，尽管是发送一个简单问候语：“你好”，RAG完整流程还是执行了，
# 检索了大量文档回来，这完全是浪费时间和资源

================User Message================
论语中教育的目的是什么？ 

=============== Context ====================
### （三）孔子及《论语》主要思想  
**教育作用**：庶、富、教；性相近，习相远。  
**教育对象**：“有教无类”，教育民主思想。  
**教育目的**：以完善人格为教育的首要目的，培养士和君子。  
**教学内容**：文、行、忠、义。  
**教学过程**：学、思、习、行。  
**教学原则**：因材施教、启发诱导、学思结合、谦虚笃实。  
**教师观**：其身正，不令而行；其身不正，虽令不从。

### （四）孟子主要思想  
**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  
**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  
**教学原则**：循序渐进，专心有恒。 

=================AI Message================
文档原文：“教育目的：以完善人格为教育的首要目的，培养士和君子。”
================User Message================
你好？ 

=============== Context ====================
### （二）经济与教育的关系  
经济发展对教育的影响：经济发展的规模和速度制约着教育发展的规模和速度；经济发展水平制约着教育事业发展的结构和布局；经济发展通过对人才和知识的需求制约着教育目标的设定；经济发展水平制约着学校的教育内容、方式方法手段和教学组织形式；经济发展制约着教育事业发展的基础。  
教育在经济发展中的作用表现为：教育为经济的发展提供良好基础；教育再生产劳动力；教育是科学技术再生产的手段。

## 第二节 教育的定义  
### （一）教育的定义  
广义的教育是指一切有目的地增进人的知识和技能，发展人的智力和体力，影响人的思想品德的社会活动，具有目的性和社会性。广义教育包括社会教育、家庭教育、学校教育。广义的教育是人类社会有史以来就有的教育活动。  
狭义的教育就是指学校教育。  
教育的要素：教育者、受教育者、教育影响（主要是教育内容）。 

=================AI